# 02 — Scripts d'extraction automatisée

**Auteur :** Benoit Girard — CheckItAI  
**Livrable n°2**

Ce notebook déroule l'**étape Extract** du pipeline : collecter les publications multimodales des trois sources, **sans intervention manuelle**, avec gestion d'erreurs et journalisation, puis sauvegarder le résultat brut en JSON.

## Définition du *done*

| Critère | Cible |
|---|---|
| Exécution sans intervention manuelle | ✅ |
| Code structuré en fonctions claires | ✅ |
| Gestion des erreurs + logs | ✅ |
| Données cohérentes avec le cas d'usage | ✅ |

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv

load_dotenv(ROOT / ".env")

from checkitai.logging_setup import setup_logging

setup_logging()

## 1. Architecture de l'extraction

Le code est **modularisé** : un module par source dans `checkitai.sources`, chacun exposant une fonction `fetch_*`. L'orchestrateur `extract_all` isole chaque source dans un `try/except` : une source en panne n'interrompt jamais les autres.

In [2]:
from checkitai.config import ExtractionConfig
from checkitai.extract import extract_all, save_raw

records = extract_all(ExtractionConfig())
print(f"Total : {len(records)} publications brutes")

2026-06-29 18:06:50 | INFO    | checkitai.extract | Extraction : demarrage de la source 'rss'


2026-06-29 18:06:50 | INFO    | checkitai.sources.rss | RSS : lecture du flux 'the_guardian' (https://www.theguardian.com/world/rss)


2026-06-29 18:06:51 | INFO    | checkitai.sources.rss | RSS : 45 publications recuperees depuis 'the_guardian'


2026-06-29 18:06:51 | INFO    | checkitai.sources.rss | RSS : lecture du flux 'bbc_news' (https://feeds.bbci.co.uk/news/world/rss.xml)


2026-06-29 18:06:52 | INFO    | checkitai.sources.rss | RSS : 38 publications recuperees depuis 'bbc_news'


...

2026-06-29 18:06:54 | INFO    | checkitai.sources.fakenewsnet | FakeNewsNet : utilisation de l'echantillon versionne fakenewsnet_sample.csv


2026-06-29 18:06:54 | INFO    | checkitai.sources.fakenewsnet | FakeNewsNet : 24 publications labellisees chargees


2026-06-29 18:06:54 | INFO    | checkitai.extract | Extraction : source 'fakenewsnet' -> 24 publications


2026-06-29 18:06:54 | INFO    | checkitai.extract | Extraction : 142 publications brutes au total


Total : 142 publications brutes


## 2. Répartition par source

On vérifie que chaque source a contribué.

In [3]:
import pandas as pd

df = pd.DataFrame(records)
df["source"].value_counts()

source
rss:the_guardian          45
rss:bbc_news              38
rss:abc_news              25
fakenewsnet:politifact    12
fakenewsnet:gossipcop     12
newsdata                  10
Name: count, dtype: int64

## 3. Vérification de la multimodalité

On contrôle la **présence de liens d'images exploitables** : un détecteur multimodal a besoin du couple texte + image.

In [4]:
avec_image = df["image_url"].astype(bool).sum()
print(f"{avec_image}/{len(df)} publications brutes possèdent une URL d'image")
df[df["image_url"].astype(bool)][["source", "title", "image_url"]].head(5)

142/142 publications brutes possèdent une URL d'image


,source,title,image_url
0,rss:the_guardian,‘Everyone is talking about Cape Verde’: World ...,https://i.guim.co.uk/img/media/9987cc89c3e8425...
1,rss:the_guardian,Whereabouts of nearly 300 people with Ebola un...,https://i.guim.co.uk/img/media/d34d9403c3a9f36...
2,rss:the_guardian,Outrage as woman jailed for three years after ...,https://i.guim.co.uk/img/media/7561cd7fe9b02aa...
3,rss:the_guardian,‘Constitutional coup’ claims as Zimbabwe senat...,https://i.guim.co.uk/img/media/ef3345b4ffa9ae8...
4,rss:the_guardian,France confirms first Ebola case in doctor who...,https://i.guim.co.uk/img/media/fa437c938c5a698...


## 4. Sauvegarde du brut

Les données brutes sont écrites en JSON dans `data/raw/`.

In [5]:
chemin = save_raw(records)
print("Fichier brut :", chemin.name)

2026-06-29 18:06:58 | INFO    | checkitai.extract | Extraction : 142 publications ecrites dans G:\Mon Drive\OC\Projet_12\checkitai\data\raw\raw_publications_20260629_160658.json


Fichier brut : raw_publications_20260629_160658.json


## Conclusion

L'extraction s'exécute d'un seul appel, journalise chaque étape et produit un JSON brut cohérent. Ces données alimentent l'étape de transformation (notebook 03).